In [1]:
import cobra
import pandas
from cobra.io import read_sbml_model, write_sbml_model
import logging
from cobra.flux_analysis import flux_variability_analysis
from cobra.flux_analysis import gapfill
from cobra.flux_analysis.loopless import add_loopless, loopless_solution
from cobra import Model, Reaction, Metabolite
from copy import deepcopy
from collections import defaultdict
from cobra.io import load_json_model

In [2]:
#Function based on https://github.com/opencobra/cobrapy/issues/707 and completely altered by me!

def removeDuplicateRxn(model):
    model2 = deepcopy(model)
    toRemove = []
    doubt = []
    
    for eachReaction in model.reactions:

        if(eachReaction.id not in toRemove):

            ids = []
            stechiometry = []
            gn = []
        
            #Placing metabolites and genes of R1 in list
            for eachMet in eachReaction.metabolites:
                ids.append(eachMet.id)
                stechiometry.append(eachReaction.metabolites[eachMet])

            ids.sort()
        
            for eachGene in eachReaction.genes:
                gn.append(eachGene.id)
    
        #Starting comparison
            for eachReaction2 in model2.reactions:
            
                if(eachReaction2.id not in toRemove):
                           
                    if eachReaction.id != eachReaction2.id: #Comparing ids to avoid self comparison
                
                        ids2=[]
                        stechiometry2 = []

                        for eachMet2 in eachReaction2.metabolites:
                            ids2.append(eachMet2.id)
                            stechiometry2.append(eachReaction2.metabolites[eachMet2])
                
                        ids2.sort()    
                
                        if(ids == ids2): #all metabolites are the same
                    
                            #Comparing genes
                            duplicate = 0
                            unsure = 0
                        
                            if(len(gn) == 0 or len(eachReaction2.genes) == 0): #one of the reactions don't have associated genes
                                unsure = 1

                            else:
                                for eachGene2 in eachReaction2.genes:
                                    if(eachGene2.id in gn): #At least one gene is the same
                                        duplicate = 1
                                        break
                        
                            if(duplicate == 1): #Checking if any reaction is reversible and removing the non-reversible one
                                if(eachReaction.lower_bound != 0 and eachReaction.upper_bound != 0): 
                                    toRemove.append(eachReaction2.id) 
                                elif(eachReaction2.lower_bound != 0 and eachReaction2.upper_bound != 0):
                                    toRemove.append(eachReaction.id)
                                else:
                                    unsure = 1
                                    
                            if(unsure == 1):
                                td=[]
                                td.append(eachReaction.id)
                                td.append(eachReaction2.id)
                                td.sort()
                                doubt.append(str(td[0]+":"+td[1]))
                                
                
    a = list(set(doubt))
    model2.remove_reactions(toRemove)
    model2.repair()
    return[model2,toRemove,a]                       

In [3]:
#Changing configuration
cobra_config = cobra.Configuration()
cobra_config.solver = "cplex"
cobra_config.bounds = -999999.0,999999.0

In [4]:
#Setting inputfile
input = "/scr/k61san/natasha/matomic/trials/CarveMe/Ecoli.tcds.top3.gramNegN.cim8.xml"

In [5]:
#Read carveme model
model = cobra.io.read_sbml_model(str(input))

In [6]:
#Fixing masses
model.metabolites.get_by_id("23dappa_c").formula = "C3H9N2O2"
model.metabolites.get_by_id("23dappa_e").formula = "C3H9N2O2"
model.metabolites.get_by_id("23dappa_p").formula = "C3H9N2O2"
model.metabolites.get_by_id("23dhb_c").formula = "C7H6O4"
model.metabolites.get_by_id("23dhbzs_c").formula = "C10H11NO6"
model.metabolites.get_by_id("23dhbzs2_c").formula = "C20H20N2O11"
model.metabolites.get_by_id("23dhbzs3_c").formula = "C30H29N3O16"
model.metabolites.get_by_id("3hmrsACP_c").formula = "C25H47N2O9PRS"
model.metabolites.get_by_id("ACP_c").formula = "C11H21N2O7PRS"
model.metabolites.get_by_id("apoACP_c").formula = "HOR"
model.metabolites.get_by_id("db4p_c").formula = "C4H7O6P"
model.metabolites.get_by_id("dcaACP_c").formula = "C21H39N2O8PRS"
model.metabolites.get_by_id("dmlz_c").formula = "C13H18N4O6"
model.metabolites.get_by_id("enter_c").formula = "C30H27N3O15"
model.metabolites.get_by_id("enter_e").formula = "C30H27N3O15"
model.metabolites.get_by_id("enter_p").formula = "C30H27N3O15"
model.metabolites.get_by_id("fad_c").formula = "C27H31N9O15P2"
model.metabolites.get_by_id("fad_e").formula = "C27H31N9O15P2"
model.metabolites.get_by_id("fad_p").formula = "C27H31N9O15P2"
model.metabolites.get_by_id("fadh2_c").formula = "C27H33N9O15P2"
model.metabolites.get_by_id("fe3dhbzs_c").formula = "C10H11NO6Fe"
model.metabolites.get_by_id("fe3dhbzs_e").formula = "C10H11NO6Fe"
model.metabolites.get_by_id("fe3dhbzs_p").formula = "C10H11NO6Fe"
model.metabolites.get_by_id("fe3dhbzs3_c").formula = "C30H29N3O16Fe"
model.metabolites.get_by_id("fe3dhbzs3_e").formula = "C30H29N3O16Fe"
model.metabolites.get_by_id("fe3dhbzs3_p").formula = "C30H29N3O16Fe"
model.metabolites.get_by_id("feenter_c").formula = "C30H27FeN3O15"
model.metabolites.get_by_id("feenter_e").formula = "C30H27FeN3O15"
model.metabolites.get_by_id("feenter_p").formula = "C30H27FeN3O15"
model.metabolites.get_by_id("fmn_c").formula = "C17H19N4O9P"
model.metabolites.get_by_id("fmn_e").formula = "C17H19N4O9P"
model.metabolites.get_by_id("fmn_p").formula = "C17H19N4O9P"
model.metabolites.get_by_id("fmnh2_c").formula = "C17H21N4O9P"
model.metabolites.get_by_id("fmnRD_c").formula = "C17H21N4O9P"
model.metabolites.get_by_id("fpram_c").formula = "C8H14N3O8P" #nao mudar
model.metabolites.get_by_id("lipoate_c").formula = "C8H14O2S2"
model.metabolites.get_by_id("lipoate_e").formula = "C8H14O2S2"
model.metabolites.get_by_id("lipoate_p").formula = "C8H14O2S2"
model.metabolites.get_by_id("myrsACP_c").formula = "C25H47N2O8PRS"
model.metabolites.get_by_id("octeACP_c").formula = "C29H53N2O8PRS"
model.metabolites.get_by_id("palmACP_c").formula = "C27H51N2O8PRS"
model.metabolites.get_by_id("rbflvrd_c").formula = "C17H22N4O6"
model.metabolites.get_by_id("ribflv_c").formula = "C17H20N4O6"
model.metabolites.get_by_id("salchs4_c").formula = "C42H42N3O25"
model.metabolites.get_by_id("salchs4_e").formula = "C42H42N3O25"
model.metabolites.get_by_id("salchs4_p").formula = "C42H42N3O25"
model.metabolites.get_by_id("salchs4fe_c").formula = "C42FeH42N3O25"
model.metabolites.get_by_id("salchs4fe_e").formula = "C42FeH42N3O25"
model.metabolites.get_by_id("salchs4fe_p").formula = "C42FeH42N3O25"
model.metabolites.get_by_id("tag__D_e").formula = "C6H12O6"
model.metabolites.get_by_id("tag__D_p").formula = "C6H12O6"
model.metabolites.get_by_id("tag1p__D_c").formula = "C6H11O9P"
model.metabolites.get_by_id("tmrs2eACP_c").formula = "C25H45N2O8PRS"

In [7]:
#Fixing charges
model.metabolites.get_by_id("1agpg161_p").charge = -1
model.metabolites.get_by_id("23dappa_c").charge = 1
model.metabolites.get_by_id("23dappa_e").charge = 1
model.metabolites.get_by_id("23dappa_p").charge = 1
model.metabolites.get_by_id("23ddhb_c").charge = -1
model.metabolites.get_by_id("23dhb_c").charge = 0
model.metabolites.get_by_id("23dhbzs_c").charge = 0
model.metabolites.get_by_id("2agpg120_c").charge = -1
model.metabolites.get_by_id("2agpg120_p").charge = -1
model.metabolites.get_by_id("2agpg180_c").charge = -1
model.metabolites.get_by_id("2agpg180_p").charge = -1
model.metabolites.get_by_id("2ahbut_c").charge = -1
model.metabolites.get_by_id("2shchc_c").charge = -2
model.metabolites.get_by_id("2dr1p_c").charge = -2
model.metabolites.get_by_id("2hdecg3p_c").charge = -1
model.metabolites.get_by_id("2hdecg3p_p").charge = -1
model.metabolites.get_by_id("2me4p_c").charge = -2
model.metabolites.get_by_id("3hmrsACP_c").charge = -1
model.metabolites.get_by_id("3hodcoa_c").charge = -4
model.metabolites.get_by_id("3hocoa_c").charge = -4
model.metabolites.get_by_id("5aizc_c").charge = -3
model.metabolites.get_by_id("6pgg_c").charge = -2
model.metabolites.get_by_id("acgam1p_c").charge = -2
model.metabolites.get_by_id("acgam1p_e").charge = -2
model.metabolites.get_by_id("acgam1p_p").charge = -2
model.metabolites.get_by_id("acmanap_c").charge = -2
model.metabolites.get_by_id("acmum6p_c").charge = -3
model.metabolites.get_by_id("ACP_c").charge = -1
model.metabolites.get_by_id("air_c").charge = -2
model.metabolites.get_by_id("anhgm3p_c").charge = -2
model.metabolites.get_by_id("anhgm3p_p").charge = -2
model.metabolites.get_by_id("apoACP_c").charge = 0
model.metabolites.get_by_id("argsuc_c").charge = -1
model.metabolites.get_by_id("crncoa_c").charge = -3
model.metabolites.get_by_id("ctbt_c").charge = 0
model.metabolites.get_by_id("ctbt_e").charge = 0
model.metabolites.get_by_id("ctbt_p").charge = 0
model.metabolites.get_by_id("ctbtcoa_c").charge = -3
model.metabolites.get_by_id("db4p_c").charge = -2
model.metabolites.get_by_id("dcamp_c").charge = -4
model.metabolites.get_by_id("dhpmp_c").charge = -2
model.metabolites.get_by_id("dscl_c").charge = -7
model.metabolites.get_by_id("enter_c").charge = 0
model.metabolites.get_by_id("enter_e").charge = 0
model.metabolites.get_by_id("enter_p").charge = 0
model.metabolites.get_by_id("fad_c").charge = -2
model.metabolites.get_by_id("fad_e").charge = -2
model.metabolites.get_by_id("fad_p").charge = -2
model.metabolites.get_by_id("fadh2_c").charge = -2
model.metabolites.get_by_id("fc1p_c").charge = -2
model.metabolites.get_by_id("fdp_c").charge = -4
model.metabolites.get_by_id("fdxrd_c").charge = 0
model.metabolites.get_by_id("fe3dcit_c").charge = -3
model.metabolites.get_by_id("fe3dcit_e").charge = -3
model.metabolites.get_by_id("fe3dcit_p").charge = -3
model.metabolites.get_by_id("fe3dhbzs_c").charge = 3
model.metabolites.get_by_id("fe3dhbzs_e").charge = 3
model.metabolites.get_by_id("fe3dhbzs_p").charge = 3
model.metabolites.get_by_id("feenter_c").charge = 3
model.metabolites.get_by_id("feenter_e").charge = 3
model.metabolites.get_by_id("feenter_p").charge = 3
model.metabolites.get_by_id("feoxam_c").charge = 1
model.metabolites.get_by_id("feoxam_e").charge = 1
model.metabolites.get_by_id("feoxam_p").charge = 1
model.metabolites.get_by_id("feoxam_un_c").charge = -2
model.metabolites.get_by_id("feoxam_un_e").charge = -2
model.metabolites.get_by_id("feoxam_un_p").charge = -2
model.metabolites.get_by_id("fgam_c").charge = -2
model.metabolites.get_by_id("fmn_c").charge = -2
model.metabolites.get_by_id("fmn_e").charge = -2
model.metabolites.get_by_id("fmn_p").charge = -2
model.metabolites.get_by_id("fmnh2_c").charge = -2
model.metabolites.get_by_id("focytc_c").charge = 1
model.metabolites.get_by_id("fpram_c").charge = -2 #nao mudar
model.metabolites.get_by_id("frulysp_c").charge = -1
model.metabolites.get_by_id("fruur_c").charge = -1
model.metabolites.get_by_id("fruur_e").charge = -1
model.metabolites.get_by_id("fruur_p").charge = -1
model.metabolites.get_by_id("g3p_c").charge = -2
model.metabolites.get_by_id("g3pg_c").charge = -1
model.metabolites.get_by_id("g3pg_e").charge = -1
model.metabolites.get_by_id("g3pg_p").charge = -1
model.metabolites.get_by_id("gdptp_c").charge = -7
model.metabolites.get_by_id("glcur1p_e").charge = -3
model.metabolites.get_by_id("glcur1p_p").charge = -3
model.metabolites.get_by_id("hkntd_c").charge = -2
model.metabolites.get_by_id("lipoate_c").charge = 0
model.metabolites.get_by_id("lipoate_e").charge = 0
model.metabolites.get_by_id("lipoate_p").charge = 0
model.metabolites.get_by_id("man1p_c").charge = -2
model.metabolites.get_by_id("man6p_c").charge = -2
model.metabolites.get_by_id("man6p_e").charge = -2
model.metabolites.get_by_id("man6p_p").charge = -2
model.metabolites.get_by_id("man6pglyc_c").charge = -3
model.metabolites.get_by_id("mmet_c").charge = 1
model.metabolites.get_by_id("mmet_e").charge = 1
model.metabolites.get_by_id("mmet_p").charge = 1
model.metabolites.get_by_id("murein3px3p_p").charge = -4
model.metabolites.get_by_id("murein4px4p_p").charge = -4
model.metabolites.get_by_id("murein5px4p_p").charge = -4
model.metabolites.get_by_id("murein5px4px4p_p").charge = -6
model.metabolites.get_by_id("myrsACP_c").charge = -1
model.metabolites.get_by_id("oc2coa_c").charge = -4
model.metabolites.get_by_id("octeACP_c").charge = -1
model.metabolites.get_by_id("op4en_c").charge = -1
model.metabolites.get_by_id("pa120_c").charge = -2
model.metabolites.get_by_id("pa120_p").charge = -2
model.metabolites.get_by_id("pa140_c").charge = -2
model.metabolites.get_by_id("pa140_p").charge = -2
model.metabolites.get_by_id("pa141_c").charge = -2
model.metabolites.get_by_id("pa141_p").charge = -2
model.metabolites.get_by_id("pa161_c").charge = -2
model.metabolites.get_by_id("pa161_p").charge = -2
model.metabolites.get_by_id("pa180_c").charge = -2
model.metabolites.get_by_id("pa180_p").charge = -2
model.metabolites.get_by_id("pa181_c").charge = -2
model.metabolites.get_by_id("pa181_p").charge = -2
model.metabolites.get_by_id("palmACP_c").charge = -1
model.metabolites.get_by_id("pep_c").charge = -3
model.metabolites.get_by_id("pg120_c").charge = -1
model.metabolites.get_by_id("pg120_p").charge = -1
model.metabolites.get_by_id("pg160_c").charge = -1
model.metabolites.get_by_id("pg160_p").charge = -1
model.metabolites.get_by_id("pg161_c").charge = -1
model.metabolites.get_by_id("pg161_p").charge = -1
model.metabolites.get_by_id("pg180_c").charge = -1
model.metabolites.get_by_id("pg180_p").charge = -1
model.metabolites.get_by_id("pgp120_c").charge = -3
model.metabolites.get_by_id("pgp120_p").charge = -3
model.metabolites.get_by_id("pgp140_c").charge = -3
model.metabolites.get_by_id("pgp140_p").charge = -3
model.metabolites.get_by_id("pgp141_c").charge = -3
model.metabolites.get_by_id("pgp141_p").charge = -3
model.metabolites.get_by_id("pgp160_c").charge = -3
model.metabolites.get_by_id("pgp160_p").charge = -3
model.metabolites.get_by_id("pgp161_c").charge = -3
model.metabolites.get_by_id("pgp161_p").charge = -3
model.metabolites.get_by_id("pgp180_c").charge = -3 
model.metabolites.get_by_id("pgp180_p").charge = -3
model.metabolites.get_by_id("pgp181_c").charge = -3 
model.metabolites.get_by_id("pgp181_p").charge = -3
model.metabolites.get_by_id("ppgpp_c").charge = -6
model.metabolites.get_by_id("pppi_c").charge = -4
model.metabolites.get_by_id("pphn_c").charge = -2
model.metabolites.get_by_id("pqq_p").charge = -3
model.metabolites.get_by_id("pqqh2_p").charge = -3
model.metabolites.get_by_id("prbamp_c").charge = -4
model.metabolites.get_by_id("prbatp_c").charge = -6
model.metabolites.get_by_id("ps160_c").charge = -1
model.metabolites.get_by_id("ps161_c").charge = -1
model.metabolites.get_by_id("ps181_c").charge = -1
model.metabolites.get_by_id("r5p_c").charge = -2
model.metabolites.get_by_id("r5p_e").charge = -2
model.metabolites.get_by_id("r5p_p").charge = -2
model.metabolites.get_by_id("rml1p_c").charge = -2
model.metabolites.get_by_id("sbt6p_c").charge = -2
model.metabolites.get_by_id("sbzcoa_c").charge = -5
model.metabolites.get_by_id("scl_c").charge = -7
model.metabolites.get_by_id("suc6p_c").charge = -2
model.metabolites.get_by_id("tag1p__D_c").charge = -2
model.metabolites.get_by_id("tag6p__D_c").charge = -2
model.metabolites.get_by_id("tagdp__D_c").charge = -4
model.metabolites.get_by_id("tagur_c").charge = -1
model.metabolites.get_by_id("td2coa_c").charge = -4
model.metabolites.get_by_id("tdecoa_c").charge = -4
model.metabolites.get_by_id("tmrs2eACP_c").charge = -1
model.metabolites.get_by_id("trnaglu_c").charge = 0
model.metabolites.get_by_id("tsul_c").charge = -2
model.metabolites.get_by_id("tsul_e").charge = -2
model.metabolites.get_by_id("tsul_p").charge = -2
model.metabolites.get_by_id("uaagmda_c").charge = -4
model.metabolites.get_by_id("uagmda_c").charge = -4
model.metabolites.get_by_id("uamr_c").charge = -3
model.metabolites.get_by_id("udcpdp_c").charge = -3
model.metabolites.get_by_id("udcpp_c").charge = -2
model.metabolites.get_by_id("udpacgal_e").charge = -2
model.metabolites.get_by_id("udpacgal_p").charge = -2

In [8]:
# Identifying and removing duplicated reactions
[md,rd, dt] = removeDuplicateRxn(model)

In [9]:
#Checking the ones in doubt
print(dt)

['COBALT2abcpp:Cobalt2abcppI', 'RIBabc:RIBabc1', '4HBZt3pp:UHBZ1t_pp', 'PHEME2abcpp:PHEMEabcpp', 'VALt2rpp:VALt3pp', 'ATPM:NTP1', 'QUIDHy:QUINDHyi']


In [10]:
#Removing the duplicated reactions that were in doubt
md.remove_reactions([md.reactions.get_by_id("QUIDHy"),md.reactions.get_by_id("QUINDH"),
                     md.reactions.get_by_id("QUINDHyi")])

md.remove_metabolites([md.metabolites.get_by_id("quin_c")])

md.repair()

In [11]:
#Running FVA again after removing reactions
md.summary(fva=0.95)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
arab__L_e,EX_arab__L_e,10,[6.485; 10],5,8.87%
arg__L_e,EX_arg__L_e,10,[-0.3754; 10],6,10.65%
asn__L_e,EX_asn__L_e,10,[3.361; 10],4,7.10%
asp__L_e,EX_asp__L_e,10,[4.398; 10],4,7.10%
ca2_e,EX_ca2_e,0.007276,[0.006912; 0.007276],0,0.00%
cl_e,EX_cl_e,0.007276,[0.006912; 0.007276],0,0.00%
cobalt2_e,EX_cobalt2_e,0.0001398,[0.0001328; 0.0001362],0,0.00%
cu2_e,EX_cu2_e,0.0009911,[0.0009415; 0.0009854],0,0.00%
cys__L_e,EX_cys__L_e,0.5497,[-1.181; 10],3,0.29%
fe2_e,EX_fe2_e,0.01001,[-9.98; 0.7244],0,0.00%


In [12]:
#Loading BiGG's universal model for gapfilling
universal = load_json_model("/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/carveme/data/generated/universal_model_cobrapy.json")

In [13]:
#Performing gapfill with universal model from Cobrapy to reduce blocked reactions
gapfill(md, universal, demand_reactions=False, iterations=10, lower_bound = 1e-9)

[[], [], [], [], [], [], [], [], [], []]

In [14]:
#Adding/Removing/Editing reactions to reduce blocked reactions

md.remove_reactions([md.reactions.get_by_id("13PPDH2")]) #Not connected to the network
md.remove_reactions([md.reactions.get_by_id("FNOR")]) #not connected to the network

md.remove_reactions([md.reactions.get_by_id("FOAMtrpp"),md.reactions.get_by_id("FOAMtex"),
                     md.reactions.get_by_id("EX_frmd_e")]) #Nothing is done
md.remove_metabolites([md.metabolites.get_by_id("frmd_c"),md.metabolites.get_by_id("frmd_p"),
                       md.metabolites.get_by_id("frmd_e")])

md.remove_reactions([md.reactions.get_by_id("GALpts")]) #nothing done

#md.add_reactions([universal.reactions.get_by_id("EX_ni2_e")])
md.reactions.get_by_id("Growth").add_metabolites({
#    md.metabolites.get_by_id("ni2_c"): -0.000307,
    md.metabolites.get_by_id("btn_c"): -2e-06})

md.remove_reactions([md.reactions.get_by_id("QUIN2tex"),md.reactions.get_by_id("EX_quin_e")])
md.remove_metabolites([md.metabolites.get_by_id("quin_e"),md.metabolites.get_by_id("quin_p")])

md.remove_reactions([md.reactions.get_by_id("St"),md.reactions.get_by_id("EX_s_e")])
md.remove_metabolites([md.metabolites.get_by_id("s_c"),md.metabolites.get_by_id("s_e")])

md.remove_reactions([md.reactions.get_by_id("TRPTA")])

md.remove_reactions([md.reactions.get_by_id("EX_aso3_e"),md.reactions.get_by_id("ASO3tex_1"),
                     md.reactions.get_by_id("ASO3tex"),md.reactions.get_by_id("ASO3t8pp")])
md.remove_metabolites([md.metabolites.get_by_id("aso3_p"),md.metabolites.get_by_id("aso3_e"),
                       md.metabolites.get_by_id("aso3_c")])

md.remove_metabolites([md.metabolites.get_by_id("gtspmd_c"),md.metabolites.get_by_id("spmd_e"),
                       md.metabolites.get_by_id("spmd_p"),md.metabolites.get_by_id("spmd_c")])
md.remove_reactions([md.reactions.get_by_id("EX_spmd_e"),md.reactions.get_by_id("SPMDtex"),
                     md.reactions.get_by_id("SPMDabc"),md.reactions.get_by_id("SPMDabcpp"),
                     md.reactions.get_by_id("SPMDt3pp"),md.reactions.get_by_id("GSPMDA"),
                     md.reactions.get_by_id("GSPMDS")])

md.remove_reactions([md.reactions.get_by_id("EX_tol_e"),md.reactions.get_by_id("TOLt6"),
                     md.reactions.get_by_id("TOLt5"),md.reactions.get_by_id("TOLtex"),
                     md.reactions.get_by_id("TOLtpp")])
md.remove_metabolites([md.metabolites.get_by_id("tol_c"),md.metabolites.get_by_id("tol_p"),
                       md.metabolites.get_by_id("tol_e")])

md.remove_metabolites([md.metabolites.get_by_id("13ppd_c"),md.metabolites.get_by_id("3hppnl_c"),
                       md.metabolites.get_by_id("dgal6p_c"),md.metabolites.get_by_id("fdxo_2_2_c"),
                       md.metabolites.get_by_id("fdxrd_c"),md.metabolites.get_by_id("indpyr_c")])              

md.repair()

In [17]:
#Running FVA again after removing reactions
md.summary(fva=0.95)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
arab__L_e,EX_arab__L_e,10,[6.485; 10],5,8.87%
arg__L_e,EX_arg__L_e,10,[-0.3754; 10],6,10.65%
asn__L_e,EX_asn__L_e,10,[3.361; 10],4,7.10%
asp__L_e,EX_asp__L_e,10,[4.398; 10],4,7.10%
btn_e,EX_btn_e,2.796E-06,[2.656E-06; 2.657E-06],10,0.00%
ca2_e,EX_ca2_e,0.007276,[0.006912; 0.007276],0,0.00%
cl_e,EX_cl_e,0.007276,[0.006912; 0.007276],0,0.00%
cobalt2_e,EX_cobalt2_e,0.0001398,[0.0001328; 0.0001362],0,0.00%
cu2_e,EX_cu2_e,0.0009911,[0.0009415; 0.0009895],0,0.00%
cys__L_e,EX_cys__L_e,0.5497,[-1.181; 10],3,0.29%


In [18]:
#Running FVA to id blocked reactions
#Universally blocked reactions are reactions that during Flux Variability Analysis cannot carry any flux while all 
#model boundaries are open. Generally blocked reactions are caused by network gaps, which can be attributed to 
#scope or knowledge gaps. 
#Loopless FBA with new set of reactions
# Trying to solve Stoichiometrically Balanced Cycles 

rlist = [md.reactions.get_by_id("5DGLCNR"),md.reactions.get_by_id("5DKGR"),md.reactions.get_by_id("ACACT6r"),
         md.reactions.get_by_id("ACACT6r_1"),md.reactions.get_by_id("ACALD"),md.reactions.get_by_id("ACALDt"),
         md.reactions.get_by_id("ACALDtex"),md.reactions.get_by_id("ACALDtpp"),md.reactions.get_by_id("ACCOAL"),
         md.reactions.get_by_id("ACOAD1f"),md.reactions.get_by_id("ACOAD1fr"),md.reactions.get_by_id("ACOAD2"),
         md.reactions.get_by_id("ACOAD2f"),md.reactions.get_by_id("ACOAD3"),md.reactions.get_by_id("ACOAD3f"),
         md.reactions.get_by_id("ACOAD4"),md.reactions.get_by_id("ACOAD4f"),md.reactions.get_by_id("ACOAD5"),
         md.reactions.get_by_id("ACOAD5_1"),md.reactions.get_by_id("ACOAD5f"),md.reactions.get_by_id("ACOAD6"),
         md.reactions.get_by_id("ACOAD6f"),md.reactions.get_by_id("ACOAD7"),md.reactions.get_by_id("ACOAD7f"),
         md.reactions.get_by_id("ADK1"),md.reactions.get_by_id("ADK3"),md.reactions.get_by_id("AGMPTRCtpp"),
         md.reactions.get_by_id("ALAR"),md.reactions.get_by_id("ALATA_D"),md.reactions.get_by_id("ALATA_L"),
         md.reactions.get_by_id("ALCD19"),md.reactions.get_by_id("ALCD19y"),md.reactions.get_by_id("ARGAGMt7pp"),
         md.reactions.get_by_id("ARGORNt7pp"),md.reactions.get_by_id("ASPT"),md.reactions.get_by_id("ASPTA"),
         md.reactions.get_by_id("ASPtpp"),md.reactions.get_by_id("ATHRDHr"),md.reactions.get_by_id("CO2t"),
         md.reactions.get_by_id("CO2tex"),md.reactions.get_by_id("CO2tpp"),md.reactions.get_by_id("DAAD"),
         md.reactions.get_by_id("DMALRED"),md.reactions.get_by_id("FADRx"),md.reactions.get_by_id("FADRx2"),
         md.reactions.get_by_id("FFSD1r"),md.reactions.get_by_id("FOMETRi"),md.reactions.get_by_id("FUM"),
         md.reactions.get_by_id("FUMt1pp"),md.reactions.get_by_id("GLBRAN2"),md.reactions.get_by_id("GLDBRAN2"),
         md.reactions.get_by_id("GLUDy"),md.reactions.get_by_id("GLUR"),md.reactions.get_by_id("GLYAT"),
         md.reactions.get_by_id("GUAt"),md.reactions.get_by_id("GUAtex"),md.reactions.get_by_id("GUAtpp"),
         md.reactions.get_by_id("H2Ot"),md.reactions.get_by_id("H2Otex"),md.reactions.get_by_id("H2Otpp"),
         md.reactions.get_by_id("H2St1"),md.reactions.get_by_id("H2St1pp"),md.reactions.get_by_id("H2Stex"),
         md.reactions.get_by_id("HPYRI"),md.reactions.get_by_id("HPYRRx"),md.reactions.get_by_id("HPYRRy"),
         md.reactions.get_by_id("IDOND"),md.reactions.get_by_id("IDOND2"),md.reactions.get_by_id("INDOLEt2pp"),
         md.reactions.get_by_id("INDOLEt2rpp"),md.reactions.get_by_id("LCARS"),md.reactions.get_by_id("LCARSyi"),
         md.reactions.get_by_id("MALt5"),md.reactions.get_by_id("MALtex"),md.reactions.get_by_id("MDH"),
         md.reactions.get_by_id("MG2tex"),md.reactions.get_by_id("MG2tpp"),md.reactions.get_by_id("MGt5"),
         md.reactions.get_by_id("MN2tipp"),md.reactions.get_by_id("MN2tpp"),md.reactions.get_by_id("NADDP"),
         md.reactions.get_by_id("NADDPp_1"),md.reactions.get_by_id("NADTRHD"),md.reactions.get_by_id("NARK"),
         md.reactions.get_by_id("NAtex"),md.reactions.get_by_id("NDPK1"),md.reactions.get_by_id("NO2tex"),
         md.reactions.get_by_id("NO3t"),md.reactions.get_by_id("NO3t7pp"),md.reactions.get_by_id("NO3tex"),
         md.reactions.get_by_id("P5CR"),md.reactions.get_by_id("PPAKr"),md.reactions.get_by_id("PPCSCT"),
         md.reactions.get_by_id("PROD2"),md.reactions.get_by_id("PSUDS"),md.reactions.get_by_id("PTA2"),
         md.reactions.get_by_id("PTRCORNt7pp"),md.reactions.get_by_id("SUCASPtpp"),
         md.reactions.get_by_id("SUCCt1pp"),md.reactions.get_by_id("SUCFUMtpp"),md.reactions.get_by_id("SUCOAS"),
         md.reactions.get_by_id("SULR"),md.reactions.get_by_id("SULR_1"),md.reactions.get_by_id("THFAT"),
         md.reactions.get_by_id("THRA2"),md.reactions.get_by_id("TRE6PH"),md.reactions.get_by_id("TRSARr"),
         md.reactions.get_by_id("VALTA"),md.reactions.get_by_id("VPAMTr"),md.reactions.get_by_id("XYLI2"),
         md.reactions.get_by_id("YUMPS"),md.reactions.get_by_id("ZN2tpp"),md.reactions.get_by_id("Zn2tex"),
         md.reactions.get_by_id("r2465_1")]

flux_variability_analysis(md,rlist,loopless=True)

,minimum,maximum
5DGLCNR,0.000000e+00,0.000000
5DKGR,0.000000e+00,0.000000
ACACT6r,0.000000e+00,0.497468
ACACT6r_1,0.000000e+00,0.497468
ACALD,-7.161003e+01,0.000000
...,...,...
XYLI2,1.360689e-11,0.000000
YUMPS,0.000000e+00,0.000000
ZN2tpp,0.000000e+00,0.000477
Zn2tex,5.541823e-11,0.000477


In [19]:
output = input.replace("xml", "")
output

'/scr/k61san/natasha/matomic/trials/CarveMe/Ecoli.tcds.top3.gramNegN.cim8.'

In [20]:
#Writting intermediate network
sbml = output + "manual.xml"
print(sbml)
cobra.io.write_sbml_model(md, sbml)

/scr/k61san/natasha/matomic/trials/CarveMe/Ecoli.tcds.top3.gramNegN.cim8.manual.xml
